# Prepare Requirements Dataset from PURE, PROMISE, DevBench, and rSDE-Bench Sources

In this notebook, the requirements datasets that come in various formats: PDF, DOC, XML, MD are preprocessed into a common JSON format which is consumable both by the RAG index and the evaluation framework.

## Import Libraries

In [1]:
import pandas as pd
import numpy as np
import json
import re

import xmltodict
import pdfplumber as pp
from docx import Document

## Prepare a Requirements Dataset
A pandas dataset with the necessary fields to collect all the requirements in the following sections.

In [5]:
# Create an empty pandas dataframe and add the required columns to it.
requirements = pd.DataFrame()

requirements['project_id'] = pd.Series(dtype='int')
requirements['project_name'] = pd.Series(dtype='str')
requirements['requirement'] = pd.Series(dtype='object')
requirements['project_description'] = pd.Series(dtype='string')

# Create the requirements tags common to all requirements
req_tags = ['Functional', 'NonFunctional', 'Quality', 'Availability', 'FaultTolerance', 'Legal', 'LookAndFeel', 'Maintainability', 'Operability', 'Performance', 'Portability', 'Scalability', 'Security', 'Usability', 'Other']

## Preprocess the XML Files from PURE
Some of the requirements documents from PURE are available as XML files. This section uses the xmltodict and re libraries to extract the requirements and append to the requirements dataset.

In [6]:
# XML preprocessing

## Preprocess PDF Files from PURE
This block of code extracts the requirements from the PDF files in the PURE requirements dataset. The libraries used are pdfplumber, and re.

In [7]:
# PDF preprocessing

## Preprocess Docx Files from PURE
Some of the requirements in PURE are in .doc format. These files are first opened in Word and saved as .docx files since the python-docx library is better at handling .docx files. The libraries used are python-docx, and re.

In [8]:
# Docx preprocessing

## Preprocess CSV File from PROMISE
The PROMISE dataset comes in a CSV file with functional and non-functional requirements and 15 labels or tags for each requirement. These set of requirements are also appended to the JSON file after preprocessing them using the pandas library.

In [8]:
# CSV preprocessing
promise = pd.read_csv('../../datasets/promise/PROMISE-relabeled-NICE.csv')

# A map to convert the promise binary field names into tags with friendly names
REQ_TYPE_MAP = {
    'IsFunctional': 'Functional',
    'IsQuality':     'Quality',
    'Availability (A)': 'Availability',
    'Fault Tolerance (FT)': 'FaultTolerance',
    'Legal (L)': 'Legal',
    'Look & Feel (LF)': 'LookAndFeel',
    'Maintainability (MN)': 'Maintainability',
    'Operability (O)': 'Operability',
    'Performance (PE)':   'Performance',
    'Portability (PO)': 'Portability',
    'Scalability (SC)': 'Scalability',
    'Security (SE)': 'Security',
    'Usability (US)': 'Usability',
    'Other (OT)': 'Other'
}


def _build_tag_row(row: pd.Series) -> list:
    """
    Return a list of friendly tag names for a single row.
    """
    tags = []
    if row['IsFunctional'] == 1:
        tags.append('Functional')
    else:
        tags.append('NonFunctional')

    for col, friendly in REQ_TYPE_MAP.items():
        if col == 'IsFunctional':
            continue
        if row[col] == 1:
            tags.append(friendly)

    return tags


def append_with_transform(source, destination) -> pd.DataFrame:
    """
    Consolidate PROMISE rows by project_id, then append to destination.

    Each project_id gets a single entry with:
      - requirement: list of {text, tag} dicts preserving per-requirement tags
    """
    source['RequirementText'] = source['RequirementText'].str.strip("'")
    source['_tags'] = source.apply(_build_tag_row, axis=1)

    # Build a list of {text, tag} dicts per project
    source['_entry'] = source.apply(
        lambda r: {'text': r['RequirementText'], 'tag': r['_tags']}, axis=1
    )

    grouped = source.groupby('ProjectID').agg(
        requirement=('_entry', list),
    ).reset_index()

    grouped = grouped.rename(columns={'ProjectID': 'project_id'})

    return pd.concat([destination, grouped[['project_id', 'requirement']]], ignore_index=True)


requirements = append_with_transform(promise, requirements)
print(f"Projects added: {requirements.shape[0]}")
print(requirements.head())


Projects added: 45
   project_id project_name                                        requirement  \
0           1          NaN  [{'text': 'The system shall refresh the displa...   
1           2          NaN  [{'text': 'The look and feel of the system sha...   
2           3          NaN  [{'text': 'The system shall be easy to use by ...   
3           4          NaN  [{'text': 'The Disputes application shall comp...   
4           5          NaN  [{'text': 'The product must support Internet E...   

  project_description  
0                <NA>  
1                <NA>  
2                <NA>  
3                <NA>  
4                <NA>  


### Infer Project Name and Description
Use an LLM to infer a short project name and one-sentence description from each project's requirements.

In [9]:
import litellm

LLM_MODEL = "ollama_chat/gpt-oss:20b"

def infer_project_metadata(reqs: list[dict]) -> dict:
    """
    Given a list of {text, tag} requirement dicts, ask an LLM to infer
    a project name and a one-sentence project description.
    """
    sample = reqs[:20]
    req_text = "\n".join(f"- {r['text']}" for r in sample)

    response = litellm.completion(
        model=LLM_MODEL,
        messages=[{
            "role": "user",
            "content": (
                "Below are software requirements from a single project.\n\n"
                f"{req_text}\n\n"
                "Based on these requirements, respond with ONLY a JSON object "
                "(no markdown, no explanation) with two keys:\n"
                '  "project_name": a short name for this project (2-5 words),\n'
                '  "project_description": a one-sentence description of what this project does.'
            ),
        }],
        temperature=0.0,
    )

    text = response.choices[0].message.content.strip()
    return json.loads(text)


for idx, row in requirements.iterrows():
    metadata = infer_project_metadata(row['requirement'])
    requirements.at[idx, 'project_name'] = metadata['project_name']
    requirements.at[idx, 'project_description'] = metadata['project_description']
    print(f"Project {row['project_id']}: {metadata['project_name']}")

print(requirements[['project_id', 'project_name', 'project_description']].to_string())


Project 1: MSEL Event Visualizer
Project 2: Realtor Mobile Suite
Project 3: Nursing Program Scheduler
Project 4: Dispute Management System
Project 5: Collision Estimator Web Tool
Project 6: Enterprise Room Scheduler
Project 7: Inventory & POS System
Project 8: PayGo Movie Hub
Project 9: Lead Wash & Score
Project 10: Periscope Battleship
Project 11: Call Scheduler
Project 12: Chicago Transaction Monitor
Project 13: CCR WCS Reporting System
Project 14: NFL Fantasy Manager
Project 15: Revenue Forecast System
Project 1: MSEL Event Visualizer
Project 2: Realtor Mobile Suite
Project 3: Nursing Program Scheduler
Project 4: Dispute Management System
Project 5: Collision Estimator Web Tool
Project 6: Enterprise Room Scheduler
Project 7: Inventory & POS System
Project 8: PayGo Movie Hub
Project 9: Lead Wash & Score
Project 10: Periscope Battleship
Project 11: Call Scheduler
Project 12: Chicago Transaction Monitor
Project 13: CCR WCS Reporting System
Project 14: NFL Fantasy Manager
Project 15: Re

## Preprocess the DevBench Dataset
The DevBench benchmarking dataset comes as a set of PRD (Product Requirements Document), Architecture, Class and Sequence diagrams. This block extracts the requirements from the PRD document and appends to the overall requirements dataset. The architecture and design files are used as is to compare agent design outputs with the DevBench designs as goldens. This block uses the pandas and re libraries for preprocessing.

## Preprocess the rSDE-Bench Dataset
The rSDE-Bench dataset contains game and website requirements in .md format. This block extracts the requirements from the documents and appends it to the overall requirements dataset. The re, and pandas libraries are used for this preprocessing.

In [12]:
# rSDE-Bench preprocessing

## Generate the Requirements JSON File
This is the final step of the preprocessing which converts the pandas dataset into JSON and writes the output to disk.

In [10]:
# Serialize the requirements dataframe to a JSON file.
output_path = '../../datasets/requirements/requirements.json'

# `indent=2` makes the file pretty‑printed.
requirements.to_json(output_path, orient='records', lines=False, indent=2)